# **Einleitung**

# Hypothesentest: Chi-Quadrat-Test im Rahmen der Eniac-Fallstudie

In diesem Notebook führen wir einen Chi-Quadrat-Test mit den Daten der Eniac-Fallstudie durch und wenden eine Post-hoc-Korrektur an, um paarweise Tests durchzuführen und den wahren Gewinner zu ermitteln.

# **Bibliotheken importieren**

In [5]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
from google.colab import drive

# Google Drive verbinden
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Formulierung der Nullhypothese und der Alternativhypothese

**Nullhypothese:** Das Button-Design hat keinen Einfluss auf die Klicks.

**Alternativhypothese:** Mindestens ein Design generiert signifikant mehr oder weniger Klicks.

## 2. Auswahl eines geeigneten Signifikanzniveaus Alpha (α)

Es wurde entschieden, dass in diesem Fall ein relativ hohes Alpha akzeptabel ist.

* Wir starten mit einem globalen Signifikanzniveau von **α = 0.05**.
* Da wir später mehrere Versionen paarweise vergleichen (6 Kombinationen), nutzen wir die Bonferroni-Korrektur (0.05 / 6), was ein strengeres Alpha von **α = 0.00833** für den Post-hoc-Test ergibt.

In [2]:
alpha = 0.05

## 3. Sammeln von zufälligen und unabhängigen Daten

Die wichtigen Informationen (Klicks auf jedes relevante Element & Besuche auf jeder Seite) sind in den Dateien verstreut. Lass sie uns sammeln. Wo sind die .csv-Dateien? 🥸

In [7]:
df_a = pd.read_csv('/content/drive/MyDrive/Eniac-ABTest/eniac_a.csv')
df_b = pd.read_csv('/content/drive/MyDrive/Eniac-ABTest/eniac_b.csv')
df_c = pd.read_csv('/content/drive/MyDrive/Eniac-ABTest/eniac_c.csv')
df_d = pd.read_csv('/content/drive/MyDrive/Eniac-ABTest/eniac_d.csv')

klicks_a = df_a.loc[df_a['Name'] == 'SHOP NOW', 'No. clicks'].values[0]
klicks_b = df_b.loc[df_b['Name'] == 'SHOP NOW', 'No. clicks'].values[0]
klicks_c = df_c.loc[df_c['Name'] == 'SEE DEALS', 'No. clicks'].values[0]
klicks_d = df_d.loc[df_d['Name'] == 'SEE DEALS', 'No. clicks'].values[0]

text_a = df_a.loc[df_a['Snapshot information'].str.contains('visits', na=False), 'Snapshot information'].values[0]
text_b = df_b.loc[df_b['Snapshot information'].str.contains('visits', na=False), 'Snapshot information'].values[0]
text_c = df_c.loc[df_c['Snapshot information'].str.contains('visits', na=False), 'Snapshot information'].values[0]
text_d = df_d.loc[df_d['Snapshot information'].str.contains('visits', na=False), 'Snapshot information'].values[0]

visits_a = int(text_a.split('visits')[0].split('•')[-1].strip())
visits_b = int(text_b.split('visits')[0].split('•')[-1].strip())
visits_c = int(text_c.split('visits')[0].split('•')[-1].strip())
visits_d = int(text_d.split('visits')[0].split('•')[-1].strip())

## 4. Berechnung des Testergebnisses

In [ ]:
print("--- TEIL 1: GLOBALER TEST ---")
daten_a = [klicks_a, visits_a - klicks_a]
daten_b = [klicks_b, visits_b - klicks_b]
daten_c = [klicks_c, visits_c - klicks_c]
daten_d = [klicks_d, visits_d - klicks_d]

daten_tabelle = [daten_a, daten_b, daten_c, daten_d]

chi2_stat, p_value, dof, expected = chi2_contingency(daten_tabelle)
print(f"Globaler P-Wert: {p_value}")

if p_value < 0.05:
    print("Ergebnis: Signifikanter Unterschied zwischen den 4 Versionen nachgewiesen.\n")
else:
    print("Ergebnis: Kein Unterschied gefunden.\n")

print("--- TEIL 2: PAARWEISE VERGLEICHE (POST-HOC) ---")
namen = ['Version A (Weiß)', 'Version B (Rot)', 'Version C (Weiß)', 'Version D (Rot)']
daten_liste = [daten_a, daten_b, daten_c, daten_d]

bonferroni_alpha = 0.05 / 6
print(f"Neues, strengeres Bonferroni-Alpha: {bonferroni_alpha:.5f}\n")

for i in range(len(namen)):
    for j in range(i + 1, len(namen)):
        name1 = namen[i]
        name2 = namen[j]
        tabelle = [daten_liste[i], daten_liste[j]]

        chi2, p_val, _, _ = chi2_contingency(tabelle)
        ergebnis = "Signifikant" if p_val < bonferroni_alpha else "NICHT signifikant"
        print(f"{name1} vs {name2} \n -> P-Wert: {p_val:.5f} ({ergebnis})\n")

## 5. Interpretation des Testergebnisses

**Auswertung der Klickrate (CTR):** Der initiale Chi-Quadrat-Test belegt, dass das Button-Design einen signifikanten Einfluss auf die Klicks hat. Der Post-hoc-Test mit Bonferroni-Korrektur zeigt jedoch, dass es zwischen den beiden besten Varianten (Version A und Version C) keinen statistisch signifikanten Unterschied in der Klickrate gibt (p-Wert: 0.46 > 0.008).

## Wie entscheiden wir, wer der Gewinner ist?

**Entscheidung über sekundäre Metriken:** Da die CTR keinen eindeutigen Sieger liefert, wird die Abbruchrate (Drop-Off Rate) betrachtet. Hier schneidet Version A mit ca. 62 % deutlich besser ab als Version C (über 70 %). Nutzer von Version A brechen den Kaufprozess nach dem Klick seltener ab.

🏆 **Der Gewinner:** Version A (Weiß „JETZT SHOPPEN“). Obwohl sie bei den reinen Klicks statistisch gleichauf mit Version C liegt, führt sie die Nutzer im Anschluss zuverlässiger durch den Conversion-Funnel.